# A1.8 · Malicious code execution

**Function A — Securing AI Architectures → CyberTravels' Architecture, and Every Risk It Carries**  ·  *Security of AI*

Builds on **[A1.7 · Identity spoofing and impersonation](https://spbreed.github.io/cyber-commons/lessons/A1.7.html)**.

| | |
|---|---|
| Tools used | Falco, gVisor, GLM-4.6, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Execute model-authored code and enumerate what the process could touch.

**Why a security engineer needs it.** Model-authored code runs with the runtime's privileges — reaching the filesystem, the network and any credential in the environment. The control it builds is: sandboxed execution (A3.2) and egress control (A3.3).

This is a **risk** lesson: it shows the failure happening before anything tries to stop it, so the control that follows is answering something you have already watched go wrong.

## 1 · The hook

Asking a model to write code is safe. Running the code it wrote is the part that is not, and most agent frameworks ship the second one enabled with the same process privileges as the framework itself.

> **At CyberTravels.** The Coding Agent writes a patch and the runtime executes it. On Alex's laptop that process can read `~/.aws`, the HR folder and the roadmap directory, because nothing said otherwise. R6.

## 2 · The framework

```
   model ---> "here is a script that does it" ---> agent runtime
                                                        |
                                            exec() on the host
                                                        v
                                        whatever the PROCESS can reach:
                                        files . network . credentials . socket

   writing the code is safe. running it is the part that is not.
```

**OWASP T11 — Unexpected RCE and Code Attacks. LLM05 — Improper Output Handling.**

Many useful agents write code and run it — that is what makes a data-analysis
agent or a coding agent worth having. The **agent_runtime** component executes
text the **model** produced, on a host, in a process.

The risk is not exotic. Model-authored code is just code, and it runs with
whatever the process has: the filesystem it can see, the network it can reach,
and every credential in its environment. There is no privilege boundary between
"the code the agent wrote to reformat a CSV" and "the code that reads
`~/.aws/credentials`", because both are strings passed to the same interpreter.

Two paths lead here, and only one involves an attacker:

**Steered.** An injection from A1.3 tells the agent to write particular code.
The runtime executes it because executing code is its job.

**Unsteered.** Nobody attacked anything. The agent wrote something plausible and
wrong — a cleanup routine with a path variable that resolves higher than
intended — and the blast radius was decided by the environment, not by intent.

That second path is worth sitting with. Most teams model this as an attack. In
practice the first incident is usually an ordinary bug with production
credentials in scope, which is why the control in A3.2 is about what the process
can *reach*, not about what the model can be persuaded to *write*.

> **Where this lands on the reference architecture.**
>
> ```
> ingress -> orchestrator -> agent_runtime -> model
>                                |              |
>                          messaging        tools / mcp
>                                |              |
>                       knowledge / memory   egress
>            identity + policy wrap every call · observability records it
> ```

## 3 · The risk, realised

What the executing process can reach, enumerated rather than assumed. Nothing below actually touches your machine — the environment is a fixture, so the lesson runs anywhere.

## 4 · The check, as a skill

CyberTravels' Coding Agent runs code it wrote. The skill enumerates what that process reaches — environment, filesystem, network — on an ordinary task first, because the ordinary task is the more persuasive half of the finding.

In [ ]:
# skills/threats/generated-code-reach-enumerator/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: generated-code-reach-enumerator
description: >-
  Enumerate what model-authored code can read, write and connect to when it
  executes — on an ordinary task and on a steered one — including process
  environment, filesystem and cloud metadata. Use when an agent runs code it
  wrote, or when sizing the runtime that code should execute in.
allowed-tools: Read, Grep, Glob, Bash
---

# The reach is the same whether the code was steered or not

An agent that executes its own code has the process's reach, not the task's.
The interesting measurement is that an **ordinary, unattacked** task already
touches everything the process can see; steering only changes what it does with
that reach, not how much of it there is.

## When to use this

Any agent with a code-execution tool, a notebook runner, a build step it
authors, or a shell. Run it before choosing a sandbox, because the output is
the requirement list for one.

## Procedure

**1 — Inventory the process environment.** Every variable visible to the
executing process. Credentials in the environment are reachable by any line of
code, and the agent did not have to look for them.

**2 — Inventory filesystem reach.** What the process can open, not what the
task needs. Include the agent's own configuration, adjacent workspaces, and any
key material mounted for another purpose.

**3 — Inventory network reach.** Resolve and attempt each destination the
process can open. The cloud metadata address is the one that turns a code
execution into a credential theft; test it explicitly.

**4 — Run the benign task and record what it touched.** This is the number that
changes the conversation: an ordinary task with no adversary reaching a private
key is a design fact, not an incident.

**5 — Run the steered task and diff.** The difference between the two is what
an attacker gains. It is usually smaller than people expect, because the
ordinary run already had everything.

## Output contract

```json
{
  "environment": {"variables": ["str"], "credential_shaped": ["str"]},
  "filesystem": {"readable": ["str"], "sensitive": ["str"]},
  "network": {"reachable": ["str"], "metadata_endpoint": true},
  "benign_run": {"touched": ["str"]},
  "steered_run": {"touched": ["str"], "gain_over_benign": ["str"]},
  "sandbox_requirements": ["str"]
}
```

## Failure modes

- **Measuring the task instead of the process.** The task is a suggestion; the
  process boundary is the control.
- **Skipping the metadata endpoint** because it is not in the code. It does not
  need to be.
- **Reporting only the steered run.** The benign run is the more persuasive
  half of the finding.
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/threats/generated-code-reach-enumerator/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
# skills/threats/generated-code-reach-enumerator/scripts/generated_code_reach_enumerator.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Enumerate what model-authored code reaches when executed, on an ordinary task and on a steered one.

This is the executable half of the `generated-code-reach-enumerator` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

# a stand-in for the process the agent's code runs inside
PROCESS_ENV = {
 "AWS_ACCESS_KEY_ID": "AKIA-EXAMPLE-NOT-REAL",
 "DATABASE_URL": "postgres://app:pw@prod-db/main",
 "HOME": "/home/agent",
}
FILESYSTEM = {"/home/agent/work/data.csv": "id,amount",
              "/home/agent/.ssh/id_ed25519": "PRIVATE KEY MATERIAL",
              "/etc/passwd": "root:x:0:0"}
NETWORK_REACHABLE = ["prod-db:5432", "169.254.169.254:80", "0.0.0.0/0"]

def execute(code):
    """The runtime runs model-authored text. Reach is decided by the process,
    not by the code's intent."""
    reached = []
    if "environ" in code:  reached += [f"env:{k}" for k in sorted(PROCESS_ENV)]
    if "open(" in code:    reached += [f"file:{p}" for p in sorted(FILESYSTEM)]
    if "connect" in code:  reached += [f"net:{h}" for h in NETWORK_REACHABLE]
    return reached

BENIGN = "rows = open('/home/agent/work/data.csv').read()"     # nobody attacked anything
STEERED = "import os; d=os.environ; connect('169.254.169.254')"

for label, code in (("ordinary bug / benign task", BENIGN),
                    ("steered by an injection", STEERED)):
    reach = execute(code)
    print(f"{label}:")
    print(f"   code   : {code[:58]}")
    print(f"   reached: {len(reach)} things")
    for r in reach[:6]:
        print(f"      {r}")
    print()

print("The benign task reached every file the process can see, including a")
print("private key it had no reason to touch. It was not attacked - the code")
print("used open(), and open() sees what the process sees.")
print()
print("Blast radius here is a property of the environment. A3.2 changes the")
print("environment; no amount of instruction changes it.")
assert any("id_ed25519" in r for r in execute(BENIGN))

## What you just proved

Model-authored code is executed against a fixture environment and the reach is enumerated: an ordinary, unattacked task touches every file the process can see including a private key, and steered code reaches the environment credentials and the cloud metadata address.

## Your turn

For one agent that executes code, list what is in its process environment right now. The credentials in that list are the blast radius of the next ordinary bug, not of the next attack.

---

**Next → [A1.9 · Injection through content the agent was asked to read](https://spbreed.github.io/cyber-commons/lessons/A1.9.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.8.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.8.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*